In [ ]:
# CELL 1: MOUNT GOOGLE DRIVE AND SET UP V3 PATHS

from google.colab import drive
drive.mount('/content/drive')

import os

# Main project folder (existing)
PROJECT_DIR = "/content/drive/MyDrive/Civic_AI_CNN"

# Existing dataset
DATASET_DIR = os.path.join(PROJECT_DIR, "dataset")
SELECTED_DIR = os.path.join(DATASET_DIR, "selected")
RAW_DIR = os.path.join(DATASET_DIR, "raw")

# NEW V3 paths (separate from V2)
SPLIT_DIR_V3 = os.path.join(DATASET_DIR, "split_v3")
MODEL_DIR_V3 = os.path.join(PROJECT_DIR, "final_model_v3_11")

MODEL_PATH_V3 = os.path.join(
    MODEL_DIR_V3, "civic_mobilenet_v3_v3_11.keras"
)

METADATA_PATH_V3 = os.path.join(
    MODEL_DIR_V3, "class_metadata_v3.json"
)

# Existing 11 classes
CLASS_NAMES = [
    'damaged_footpath',
    'damaged_road_sign',
    'electrical_hazard',
    'infrastructure',
    'normal',
    'open_manhole_borewell',
    'road_damage',
    'roadside_garbage',
    'streetlight_faults',
    'vegetation',
    'waterlogging'
]

NUM_CLASSES = len(CLASS_NAMES)

print("Drive mounted successfully!")
print("-" * 50)
print("Project:", PROJECT_DIR)
print("Selected dataset:", SELECTED_DIR)
print("V3 split location:", SPLIT_DIR_V3)
print("V3 model folder:", MODEL_DIR_V3)
print("V3 model filename:", MODEL_PATH_V3)
print("Number of classes:", NUM_CLASSES)

# Verify existing folders
print("\nFolder checks:")
print("Project folder exists:", os.path.exists(PROJECT_DIR))
print("Selected dataset exists:", os.path.exists(SELECTED_DIR))
print("Raw dataset exists:", os.path.exists(RAW_DIR))

Mounted at /content/drive
Drive mounted successfully!
--------------------------------------------------
Project: /content/drive/MyDrive/Civic_AI_CNN
Selected dataset: /content/drive/MyDrive/Civic_AI_CNN/dataset/selected
V3 split location: /content/drive/MyDrive/Civic_AI_CNN/dataset/split_v3
V3 model folder: /content/drive/MyDrive/Civic_AI_CNN/final_model_v3_11
V3 model filename: /content/drive/MyDrive/Civic_AI_CNN/final_model_v3_11/civic_mobilenet_v3_v3_11.keras
Number of classes: 11

Folder checks:
Project folder exists: True
Selected dataset exists: True
Raw dataset exists: True


In [ ]:
# CELL 2: CHECK DATASET IMAGE COUNTS

import os

IMAGE_EXTENSIONS = (".jpg", ".jpeg", ".png", ".webp")

def get_images(folder):
    if not os.path.isdir(folder):
        return []
    return [
        f for f in os.listdir(folder)
        if f.lower().endswith(IMAGE_EXTENSIONS)
    ]

print("=" * 55)
print("V3 DATASET VERIFICATION")
print("=" * 55)

total_images = 0

for class_name in CLASS_NAMES:
    class_path = os.path.join(SELECTED_DIR, class_name)
    images = get_images(class_path)
    count = len(images)
    total_images += count

    print(f"{class_name:30s} : {count}")

print("-" * 55)
print("TOTAL SELECTED IMAGES:", total_images)

# Check newly added raw borewell images
RAW_BOREWELL_DIR = os.path.join(RAW_DIR, "open_manhole_borewell")

print("\nRAW BOREWELL FOLDER")
print("-" * 55)
print("Folder exists:", os.path.exists(RAW_BOREWELL_DIR))
print("Raw images:", len(get_images(RAW_BOREWELL_DIR)))

# Check selected borewell images
SELECTED_BOREWELL_DIR = os.path.join(
    SELECTED_DIR, "open_manhole_borewell"
)

print("\nSELECTED BOREWELL FOLDER")
print("-" * 55)
print("Selected images:", len(get_images(SELECTED_BOREWELL_DIR)))

V3 DATASET VERIFICATION
damaged_footpath               : 200
damaged_road_sign              : 350
electrical_hazard              : 120
infrastructure                 : 199
normal                         : 249
open_manhole_borewell          : 167
road_damage                    : 393
roadside_garbage               : 120
streetlight_faults             : 248
vegetation                     : 247
waterlogging                   : 199
-------------------------------------------------------
TOTAL SELECTED IMAGES: 2492

RAW BOREWELL FOLDER
-------------------------------------------------------
Folder exists: True
Raw images: 0

SELECTED BOREWELL FOLDER
-------------------------------------------------------
Selected images: 167


In [ ]:
# CELL 3: CREATE NEW V3 TRAIN / VALIDATION / TEST SPLIT

import os
import shutil
import random

random.seed(42)

# Split proportions
TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
TEST_RATIO = 0.15

# Safety check: never overwrite an existing V3 split
if os.path.exists(SPLIT_DIR_V3) and os.listdir(SPLIT_DIR_V3):
    raise Exception(
        "split_v3 already contains files! "
        "Stopping to prevent overwriting."
    )

# Create split folders
for split in ["train", "validation", "test"]:
    os.makedirs(os.path.join(SPLIT_DIR_V3, split), exist_ok=True)

total_split_images = 0

for class_name in CLASS_NAMES:

    source_folder = os.path.join(SELECTED_DIR, class_name)

    images = [
        f for f in os.listdir(source_folder)
        if f.lower().endswith(IMAGE_EXTENSIONS)
    ]

    random.shuffle(images)

    total = len(images)
    train_count = int(total * TRAIN_RATIO)
    val_count = int(total * VAL_RATIO)

    train_images = images[:train_count]
    val_images = images[train_count:train_count + val_count]
    test_images = images[train_count + val_count:]

    for split_name, split_images in [
        ("train", train_images),
        ("validation", val_images),
        ("test", test_images)
    ]:

        destination = os.path.join(
            SPLIT_DIR_V3, split_name, class_name
        )
        os.makedirs(destination, exist_ok=True)

        for img in split_images:
            shutil.copy2(
                os.path.join(source_folder, img),
                os.path.join(destination, img)
            )

    total_split_images += total

    print(
        f"{class_name:30s} | "
        f"Train: {len(train_images):3d} | "
        f"Val: {len(val_images):3d} | "
        f"Test: {len(test_images):3d}"
    )

print("\n" + "=" * 65)
print("V3 SPLIT CREATED SUCCESSFULLY!")
print("Total images split:", total_split_images)
print("Saved at:", SPLIT_DIR_V3)

damaged_footpath               | Train: 140 | Val:  30 | Test:  30
damaged_road_sign              | Train: 244 | Val:  52 | Test:  54
electrical_hazard              | Train:  84 | Val:  18 | Test:  18
infrastructure                 | Train: 139 | Val:  29 | Test:  31
normal                         | Train: 174 | Val:  37 | Test:  38
open_manhole_borewell          | Train: 116 | Val:  25 | Test:  26
road_damage                    | Train: 275 | Val:  58 | Test:  60
roadside_garbage               | Train:  84 | Val:  18 | Test:  18
streetlight_faults             | Train: 173 | Val:  37 | Test:  38
vegetation                     | Train: 172 | Val:  37 | Test:  38
waterlogging                   | Train: 139 | Val:  29 | Test:  31

V3 SPLIT CREATED SUCCESSFULLY!
Total images split: 2492
Saved at: /content/drive/MyDrive/Civic_AI_CNN/dataset/split_v3


In [ ]:
# CELL 4: VERIFY V3 SPLIT

import os

SPLITS = ["train", "validation", "test"]

grand_total = 0

print("=" * 75)
print("V3 SPLIT VERIFICATION")
print("=" * 75)

for split in SPLITS:

    split_path = os.path.join(SPLIT_DIR_V3, split)
    split_total = 0

    print(f"\n{split.upper()}")

    for class_name in CLASS_NAMES:

        class_path = os.path.join(split_path, class_name)

        count = len([
            f for f in os.listdir(class_path)
            if f.lower().endswith(IMAGE_EXTENSIONS)
        ])

        split_total += count
        print(f"{class_name:30s}: {count}")

    grand_total += split_total
    print(f"TOTAL {split.upper()}: {split_total}")

print("\n" + "=" * 75)
print("GRAND TOTAL:", grand_total)
print("=" * 75)

V3 SPLIT VERIFICATION

TRAIN
damaged_footpath              : 140
damaged_road_sign             : 244
electrical_hazard             : 84
infrastructure                : 139
normal                        : 174
open_manhole_borewell         : 116
road_damage                   : 275
roadside_garbage              : 84
streetlight_faults            : 173
vegetation                    : 172
waterlogging                  : 139
TOTAL TRAIN: 1740

VALIDATION
damaged_footpath              : 30
damaged_road_sign             : 52
electrical_hazard             : 18
infrastructure                : 29
normal                        : 37
open_manhole_borewell         : 25
road_damage                   : 58
roadside_garbage              : 18
streetlight_faults            : 37
vegetation                    : 37
waterlogging                  : 29
TOTAL VALIDATION: 370

TEST
damaged_footpath              : 30
damaged_road_sign             : 54
electrical_hazard             : 18
infrastructure               

In [ ]:
# CELL 5: IMPORT LIBRARIES AND CHECK GPU

import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import json
import os
import shutil

from tensorflow.keras.applications import MobileNetV3Large
from tensorflow.keras import layers, models

print("TensorFlow version:", tf.__version__)

gpus = tf.config.list_physical_devices('GPU')

if gpus:
    print("\nGPU AVAILABLE!")
    for gpu in gpus:
        print(gpu)
else:
    print("\nGPU NOT AVAILABLE - Training will use CPU.")

# Create new V3 model folder
os.makedirs(MODEL_DIR_V3, exist_ok=True)

print("\nV3 model folder ready:")
print(MODEL_DIR_V3)

print("\nNumber of classes:", NUM_CLASSES)
print("Class names:", CLASS_NAMES)

TensorFlow version: 2.21.0

GPU AVAILABLE!
PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')

V3 model folder ready:
/content/drive/MyDrive/Civic_AI_CNN/final_model_v3_11

Number of classes: 11
Class names: ['damaged_footpath', 'damaged_road_sign', 'electrical_hazard', 'infrastructure', 'normal', 'open_manhole_borewell', 'road_damage', 'roadside_garbage', 'streetlight_faults', 'vegetation', 'waterlogging']


In [ ]:
# CELL 6: LOAD V3 DATASETS

IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32
SEED = 42

TRAIN_DIR = os.path.join(SPLIT_DIR_V3, "train")
VAL_DIR = os.path.join(SPLIT_DIR_V3, "validation")
TEST_DIR = os.path.join(SPLIT_DIR_V3, "test")

train_ds = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="int",
    class_names=CLASS_NAMES,
    shuffle=True,
    seed=SEED
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    VAL_DIR,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="int",
    class_names=CLASS_NAMES,
    shuffle=False
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    TEST_DIR,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="int",
    class_names=CLASS_NAMES,
    shuffle=False
)

print("\nDataset loading completed!")

print("Training batches:", tf.data.experimental.cardinality(train_ds).numpy())
print("Validation batches:", tf.data.experimental.cardinality(val_ds).numpy())
print("Testing batches:", tf.data.experimental.cardinality(test_ds).numpy())

print("\nClass order:", train_ds.class_names)

Found 1736 files belonging to 11 classes.
Found 370 files belonging to 11 classes.
Found 382 files belonging to 11 classes.

Dataset loading completed!
Training batches: 55
Validation batches: 12
Testing batches: 12

Class order: ['damaged_footpath', 'damaged_road_sign', 'electrical_hazard', 'infrastructure', 'normal', 'open_manhole_borewell', 'road_damage', 'roadside_garbage', 'streetlight_faults', 'vegetation', 'waterlogging']


In [ ]:
# CELL 7: FIND CORRUPTED OR UNREADABLE TRAINING IMAGES

from PIL import Image

corrupted_images = []

for class_name in CLASS_NAMES:
    folder = os.path.join(TRAIN_DIR, class_name)

    for filename in os.listdir(folder):
        if filename.lower().endswith(IMAGE_EXTENSIONS):
            filepath = os.path.join(folder, filename)

            try:
                with Image.open(filepath) as img:
                    img.verify()
            except Exception as e:
                corrupted_images.append((class_name, filename, str(e)))

print("Corrupted images found:", len(corrupted_images))

for class_name, filename, error in corrupted_images:
    print(f"{class_name} | {filename} | {error}")

if not corrupted_images:
    print("No corrupted images detected by PIL.")

Corrupted images found: 0
No corrupted images detected by PIL.


In [ ]:
# CELL 8: BUILD V3 MOBILENETV3 MODEL

base_model = MobileNetV3Large(
    input_shape=(224, 224, 3),
    include_top=False,
    weights="imagenet",
    include_preprocessing=True
)

# Freeze pretrained layers initially
base_model.trainable = False

inputs = tf.keras.Input(shape=(224, 224, 3))

x = base_model(inputs, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)

outputs = layers.Dense(
    NUM_CLASSES,
    activation="softmax",
    name="classification_head"
)(x)

model = models.Model(
    inputs=inputs,
    outputs=outputs,
    name="civic_mobilenet_v3_v3_11"
)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

12683000/12683000 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "civic_mobilenet_v3_v3_11"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ MobileNetV3Large (Functional)   │ (None, 7, 7, 960)      │     2,996,352 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 960)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 960)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ classification_head (Dense)     │ (None, 11)             │        10,571 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,006,923 (11.47 MB)

 Trainable params: 10,571 (41.29 KB)

 Non-trainable params: 2,996,352 (11.43 MB)

In [ ]:
# CELL 9: TRAINING CALLBACKS

from tensorflow.keras.callbacks import (
    ModelCheckpoint,
    EarlyStopping,
    ReduceLROnPlateau
)

callbacks = [
    ModelCheckpoint(
        filepath=MODEL_PATH_V3,
        monitor="val_accuracy",
        save_best_only=True,
        mode="max",
        verbose=1
    ),

    EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True,
        verbose=1
    ),

    ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.2,
        patience=2,
        min_lr=1e-6,
        verbose=1
    )
]

print("V3 callbacks configured!")
print("Best model will be saved at:", MODEL_PATH_V3)

V3 callbacks configured!
Best model will be saved at: /content/drive/MyDrive/Civic_AI_CNN/final_model_v3_11/civic_mobilenet_v3_v3_11.keras


In [ ]:
# CELL 10: TRAIN V3 MODEL WITH LIVE EPOCH PROGRESS

EPOCHS = 22

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks,
    verbose=1
)

Epoch 1/22
55/55 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.3061 - loss: 2.2463   
Epoch 1: val_accuracy improved from None to 0.82973, saving model to /content/drive/MyDrive/Civic_AI_CNN/final_model_v3_11/civic_mobilenet_v3_v3_11.keras

Epoch 1: finished saving model to /content/drive/MyDrive/Civic_AI_CNN/final_model_v3_11/civic_mobilenet_v3_v3_11.keras
55/55 ━━━━━━━━━━━━━━━━━━━━ 141s 2s/step - accuracy: 0.4747 - loss: 1.6414 - val_accuracy: 0.8297 - val_loss: 0.6250 - learning_rate: 0.0010
Epoch 2/22
54/55 ━━━━━━━━━━━━━━━━━━━━ 0s 752ms/step - accuracy: 0.7979 - loss: 0.6407
Epoch 2: val_accuracy improved from 0.82973 to 0.88919, saving model to /content/drive/MyDrive/Civic_AI_CNN/final_model_v3_11/civic_mobilenet_v3_v3_11.keras

Epoch 2: finished saving model to /content/drive/MyDrive/Civic_AI_CNN/final_model_v3_11/civic_mobilenet_v3_v3_11.keras
55/55 ━━━━━━━━━━━━━━━━━━━━ 49s 885ms/step - accuracy: 0.8191 - loss: 0.5718 - val_accuracy: 0.8892 - val_loss: 0.3974 - learning_rate: 0.

In [ ]:
# CELL 11: LOAD BEST V3 MODEL AND TEST

from tensorflow.keras.models import load_model

# Load best saved checkpoint
best_model = load_model(MODEL_PATH_V3)

# Evaluate on test dataset
test_loss, test_accuracy = best_model.evaluate(test_ds, verbose=1)

print("\n" + "=" * 50)
print("V3 TEST RESULTS")
print("=" * 50)

print(f"Test Accuracy: {test_accuracy * 100:.2f}%")
print(f"Test Loss: {test_loss:.4f}")

print("\nModel:", MODEL_PATH_V3)

12/12 ━━━━━━━━━━━━━━━━━━━━ 40s 3s/step - accuracy: 0.9555 - loss: 0.1356

V3 TEST RESULTS
Test Accuracy: 95.55%
Test Loss: 0.1356

Model: /content/drive/MyDrive/Civic_AI_CNN/final_model_v3_11/civic_mobilenet_v3_v3_11.keras


In [ ]:
# CELL 12: CLASSIFICATION REPORT

from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

y_true = np.concatenate([
    labels.numpy() for images, labels in test_ds
])

y_prob = best_model.predict(test_ds, verbose=1)
y_pred = np.argmax(y_prob, axis=1)

print(classification_report(
    y_true,
    y_pred,
    target_names=CLASS_NAMES,
    digits=4,
    zero_division=0
))

print("\nBOREWELL CLASS PERFORMANCE")

borewell_index = CLASS_NAMES.index("open_manhole_borewell")

borewell_mask = (y_true == borewell_index)

correct = np.sum(
    (y_pred == borewell_index) & borewell_mask
)

total = np.sum(borewell_mask)

print("Correct predictions:", correct)
print("Total borewell test images:", total)
print("Borewell recall:", round(correct / total * 100, 2), "%")

12/12 ━━━━━━━━━━━━━━━━━━━━ 20s 740ms/step
                       precision    recall  f1-score   support

     damaged_footpath     1.0000    1.0000    1.0000        30
    damaged_road_sign     0.9811    0.9630    0.9720        54
    electrical_hazard     0.9444    0.9444    0.9444        18
       infrastructure     0.8529    0.9355    0.8923        31
               normal     0.9268    1.0000    0.9620        38
open_manhole_borewell     1.0000    0.8846    0.9388        26
          road_damage     0.9516    0.9833    0.9672        60
     roadside_garbage     1.0000    0.8889    0.9412        18
   streetlight_faults     0.9697    0.8421    0.9014        38
           vegetation     0.9500    1.0000    0.9744        38
         waterlogging     0.9688    1.0000    0.9841        31

             accuracy                         0.9555       382
            macro avg     0.9587    0.9493    0.9525       382
         weighted avg     0.9574    0.9555    0.9552       382


BOREWELL 

In [ ]:
# CELL 13: SAVE V3 CLASS METADATA

import json
import os

metadata_v3 = {
    "model_name": "civic_mobilenet_v3_v3_11",
    "version": "v3",
    "num_classes": NUM_CLASSES,
    "image_size": [224, 224],
    "class_names": CLASS_NAMES,
    "class_indices": {
        class_name: index
        for index, class_name in enumerate(CLASS_NAMES)
    },
    "test_accuracy": float(test_accuracy),
    "test_loss": float(test_loss)
}

with open(METADATA_PATH_V3, "w") as f:
    json.dump(metadata_v3, f, indent=4)

print("V3 metadata saved successfully!")
print("Location:", METADATA_PATH_V3)

V3 metadata saved successfully!
Location: /content/drive/MyDrive/Civic_AI_CNN/final_model_v3_11/class_metadata_v3.json


In [ ]:
# CELL 14: VERIFY V3 MODEL AND METADATA

for path in [MODEL_PATH_V3, METADATA_PATH_V3]:

    print("\nFile:", os.path.basename(path))
    print("Exists:", os.path.exists(path))

    if os.path.exists(path):
        size_mb = os.path.getsize(path) / (1024 * 1024)
        print("Size:", round(size_mb, 2), "MB")

print("\nV3 MODEL VERIFICATION COMPLETED!")


File: civic_mobilenet_v3_v3_11.keras
Exists: True
Size: 12.21 MB

File: class_metadata_v3.json
Exists: True
Size: 0.0 MB

V3 MODEL VERIFICATION COMPLETED!


In [ ]:
from google.colab import files
from tensorflow.keras.models import load_model
from tensorflow.keras.utils import load_img, img_to_array
import numpy as np
import os

# Load the saved V3 model
test_model = load_model(MODEL_PATH_V3)

# Upload an image
uploaded = files.upload()

for filename in uploaded.keys():
    image = load_img(filename, target_size=(224, 224))
    image_array = img_to_array(image)
    image_array = np.expand_dims(image_array, axis=0)

    # Predict
    predictions = test_model.predict(image_array, verbose=0)

    predicted_index = np.argmax(predictions[0])
    predicted_class = CLASS_NAMES[predicted_index]
    confidence = predictions[0][predicted_index] * 100

    print("\nImage:", filename)
    print("Predicted Category:", predicted_class)
    print("Confidence:", round(confidence, 2), "%")

Saving water-well-drilling-dig-inside-260nw-1448232047.webp to water-well-drilling-dig-inside-260nw-1448232047.webp

Image: water-well-drilling-dig-inside-260nw-1448232047.webp
Predicted Category: open_manhole_borewell
Confidence: 99.66 %


In [ ]:
from google.colab import files
from tensorflow.keras.models import load_model
from tensorflow.keras.utils import load_img, img_to_array
import numpy as np
import os

# Load the saved V3 model
test_model = load_model(MODEL_PATH_V3)

# Upload an image
uploaded = files.upload()

for filename in uploaded.keys():
    image = load_img(filename, target_size=(224, 224))
    image_array = img_to_array(image)
    image_array = np.expand_dims(image_array, axis=0)

    # Predict
    predictions = test_model.predict(image_array, verbose=0)

    predicted_index = np.argmax(predictions[0])
    predicted_class = CLASS_NAMES[predicted_index]
    confidence = predictions[0][predicted_index] * 100

    print("\nImage:", filename)
    print("Predicted Category:", predicted_class)
    print("Confidence:", round(confidence, 2), "%")

Saving 67221385.avif to 67221385.avif

Image: 67221385.avif
Predicted Category: road_damage
Confidence: 83.25 %
